In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# xgboost is NOT a component of Python, nor can import.  Must be INSTALL on local and import after that
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [2]:
# -----------------------------
# Load Data, and see what it is look like first
# -----------------------------
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')


print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train OBJECT columns", train_df.select_dtypes(include=['object']).columns)         


Train shape: (4209, 378)
Test shape: (4209, 377)
Train OBJECT columns Index(['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X8'], dtype='str')


C:\Users\SETUP\AppData\Local\Temp\ipykernel_19116\4147440609.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print("Train OBJECT columns", train_df.select_dtypes(include=['object']).columns)


In [3]:
# -----------------------------
# Identify & Remove Zero-Variance Columns
# -----------------------------
# Compute variance for all numeric columns

    # From the DataFrame train_df, select only the columns whose data type is numeric (like integers or floats), then take just the names of those columns, and assign those names to the variable numeric_cols
    # select_dtypes is a pandas DataFrame method that filters columns based on their data type (e.g., int64, float64, object, datetime).  It returns a new DataFrame containing only the columns whose data types match the criteria specify.
    # It returns a new DataFrame containing only the columns whose data types match the criteria specify.
numeric_cols = train_df.select_dtypes(include=[np.number]).columns

    # any columns with variance == 0, store it in the variable zero_var_cols
zero_var_cols = train_df[numeric_cols].var() == 0

    # zero_var_cols[zero_var_cols] check if the zero_var_cols == 0 or TRUE columns exist -> grab the name -> put to a list and assign to variable cols_to_drop
cols_to_drop = zero_var_cols[zero_var_cols].index.tolist()
print("Columns with 0_var", train_df[cols_to_drop])



Columns with 0_var       X11  X93  X107  X233  X235  X268  X289  X290  X293  X297  X330  X347
0       0    0     0     0     0     0     0     0     0     0     0     0
1       0    0     0     0     0     0     0     0     0     0     0     0
2       0    0     0     0     0     0     0     0     0     0     0     0
3       0    0     0     0     0     0     0     0     0     0     0     0
4       0    0     0     0     0     0     0     0     0     0     0     0
...   ...  ...   ...   ...   ...   ...   ...   ...   ...   ...   ...   ...
4204    0    0     0     0     0     0     0     0     0     0     0     0
4205    0    0     0     0     0     0     0     0     0     0     0     0
4206    0    0     0     0     0     0     0     0     0     0     0     0
4207    0    0     0     0     0     0     0     0     0     0     0     0
4208    0    0     0     0     0     0     0     0     0     0     0     0

[4209 rows x 12 columns]


In [4]:
# Also check non-numeric (e.g., constant strings columns)
constant_non_numeric = []
for col in train_df.columns:

    # If a column's .nunique() is equal to 1, it means that every single row in that column contains the exact same value. This is called a constant column (or a feature with zero variance).
    if train_df[col].nunique() == 1:
        constant_non_numeric.append(col)

    # add constant_non_numeric columns to numeric columns to drop them at same time 
cols_to_drop += constant_non_numeric

    # dedupe
    # A set() in Python is a collection of items that cannot have duplicates. Every element inside a set must be unique.
    # Example: If cols_to_drop = ['X1', 'X2', 'X1', 'X3']
    # Running set(cols_to_drop) turns it into {'X1', 'X2', 'X3'}.
cols_to_drop = list(set(cols_to_drop))

    # Normally in pandas, when you do something to a DataFrame, it does NOT change the original — it creates a brand new copy with the changes applied:

    # This does NOT change train_df at all
    # train_df.drop(columns=cols_to_drop)

    # To keep the result, you'd have to reassign it
    # train_df = train_df.drop(columns=cols_to_drop)

    # With inplace=True — what your code does is
    # It tells pandas:
            # "Don't make a copy — go into the original DataFrame and make the change directly there."
train_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns], inplace=True)


In [5]:
# -----------------------------
# Check Nulls & Unique Values
# -----------------------------
print("\n--- NULL CHECK ---")

    # First .sum() → "how many absent per class?"
    # Second .sum() → "how many absent in the whole school?"
print("TRAIN nulls:", train_df.isnull().sum().sum())
print("TEST nulls:", test_df.isnull().sum().sum())

print("\n--- UNIQUE VALUE EXAMPLES (first 5 cols) ---")
for col in train_df.columns[:5]:
    print(f"{col}: {train_df[col].nunique()} unique values")



--- NULL CHECK ---
TRAIN nulls: 0
TEST nulls: 0

--- UNIQUE VALUE EXAMPLES (first 5 cols) ---
ID: 4209 unique values
y: 2545 unique values
X0: 47 unique values
X1: 27 unique values
X2: 44 unique values


In [6]:
# -----------------------------
# Separate Target & Features
#
# SOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO  IMPORTANT   #########################################
# -----------------------------

y_train = train_df['y']
X_train = train_df.drop(columns=['y'])
X_test = test_df.copy()


"""          
            The big picture first
                Before training any ML model, you need to split your data into two things:

                What you want to predict → called the target (or label)
                What you use to make the prediction → called the features (or inputs)

                Think of it like a test at school:

                The features are the questions
                The target is the answer key


                Line by line breakdown

                Line 1 — Pull out the target (what you want to predict)
                python y_train = train_df['y']
                This grabs the single column named 'y' from your training data and stores it separately.
                train_df (before):
                age   salary   experience   y
                25    50000        2        1
                30    60000        5        0
                35    70000        8        1

                y_train becomes just:
                1
                0
                1
                y is the answer your model is trying to learn to predict. By convention it's always called y in ML code.

                Line 2 — Pull out the features (what you predict FROM)
                python X_train = train_df.drop(columns=['y'])
                This takes train_df and removes the target column, leaving only the input features. Everything except y becomes your X_train.
                X_train becomes:
                age   salary   experience
                25    50000        2
                30    60000        5
                35    70000        8
                X is by convention always uppercase in ML code because it represents a matrix (multiple columns), while y is lowercase because it's a single column (a vector).

                Line 3 — Copy the test data
                python X_test = test_df.copy()
                This makes a full independent copy of your test data. Notice:

                There is no y_test here — because in a real prediction scenario, the test data has no answers yet. That's what your model is going to figure out.
                .copy() is used instead of just X_test = test_df to make sure any changes to X_test later don't accidentally affect the original test_df.


                The full picture — visually
                Original train_df
                ┌─────┬────────┬────────────┬───┐
                │ age │ salary │ experience │ y │
                ├─────┼────────┼────────────┼───┤
                │ 25  │ 50000  │     2      │ 1 │
                │ 30  │ 60000  │     5      │ 0 │
                │ 35  │ 70000  │     8      │ 1 │
                └─────┴────────┴────────────┴───┘
                        │                   │
                        ▼                   ▼
                    X_train             y_train
                (features/inputs)    (target/answers)
                age, salary, exp       1, 0, 1


                Original test_df                 X_test
                ┌─────┬────────┬────────────┐    (copy — no y column,
                │ age │ salary │ experience │ →   because we don't know
                │ 28  │ 55000  │     3      │     the answers yet)
                └─────┴────────┴────────────┘

                Why this separation matters
                
                The model learns the relationship between X_train and y_train, then applies what it learned to X_test to generate predictions. That's the entire foundation of supervised machine learning.
                   
"""

"          \n            The big picture first\n                Before training any ML model, you need to split your data into two things:\n\n                What you want to predict → called the target (or label)\n                What you use to make the prediction → called the features (or inputs)\n\n                Think of it like a test at school:\n\n                The features are the questions\n                The target is the answer key\n\n\n                Line by line breakdown\n\n                Line 1 — Pull out the target (what you want to predict)\n                python y_train = train_df['y']\n                This grabs the single column named 'y' from your training data and stores it separately.\n                train_df (before):\n                age   salary   experience   y\n                25    50000        2        1\n                30    60000        5        0\n                35    70000        8        1\n\n                y_train becomes just:\n          

In [7]:
# -----------------------------
# Label Encoding for Categorical Columns
# -----------------------------
# Identify object-type columns (assumed to be categorical) ???? Sure?


cat_cols = X_train.select_dtypes(include=['object']).columns

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    





    # axis=0 to stack X_train and X_test on top of each other
    
    # Fit on union of train + test to avoid unseen label error

    """      
                In scikit-learn, the fit() method is the "learning" or "training" phase.

                Specifically for a LabelEncoder, le.fit(all_vals) scans through all the unique text strings in all_vals, sorts them alphabetically, and assigns a unique integer (0, 1, 2, 3...) to each one. It builds an internal dictionary (a mapping) that it will use later to transform those strings into numbers.
        
                1. The Setup (An Example)Imagine the loop is currently looking at a column named color. After stacking the training and test sets with axis=0, your all_vals variable looks like this:['red', 'blue', 'green', 'blue', 'red']
                
                2. What le.fit(all_vals) does under the hood when you run .fit(all_vals), the encoder does two things:It finds all the unique categories: ['blue', 'green', 'red'].It alphabetizes them and maps them to numbers starting from 0:'blue' -> 0' ; green' ->  1 , 'red' -> 2

                Why fit on both train and test combined?
                If you only ran le.fit(X_train[col]), and your test set (X_test) contained a brand new category (for example, 'yellow') that wasn't in the training set, your code would crash later during le.transform(X_test) because the encoder wouldn't know what number to give to 'yellow'.

                Fitting on all_vals ensures the encoder learns every possible category across your entire dataset ahead of time!
    """



    all_vals = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    le.fit(all_vals)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

print(f"\nEncoded {len(cat_cols)} categorical columns.")

C:\Users\SETUP\AppData\Local\Temp\ipykernel_19116\2670950000.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns



Encoded 8 categorical columns.


In [8]:
# -----------------------------
# Dimensionality Reduction (PCA)
# What PCA actually does:
# PCA (Principal Component Analysis) finds the directions in your data that carry the most information, and keeps only those.
#
# -----------------------------
# Optional: Only apply if too many features (>1000)

if X_train.shape[1] > 500:
    print("Applying PCA...")
    
        # retain 95% variance
    pca = PCA(n_components=0.95)  
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    print(f"Reduced from {X_train.shape[1]} to {X_train_pca.shape[1]} features.")
else:
    print("Skipping PCA (feature count <= 500)")
    X_train_pca = X_train.values
    X_test_pca = X_test.values

Skipping PCA (feature count <= 500)


In [9]:
# -----------------------------
# Train XGBoost & Predict
#
# XGBRegressor is a machine learning model built to predict continuous numbers (like test time, price, or temperature)
# It uses Gradient Boosting: It builds hundreds of small decision trees one after another.
# Each new tree looks at the mistakes the previous trees made and tries to fix them.
# Over time, the combined trees form a highly accurate predictor.
#
# -----------------------------
print("\nTraining XGBoost...")
model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='rmse'
)

# model.fit(X_train_pca, y_train)  |  Trains the model. It studies your training data, finds patterns, and builds the decision trees
model.fit(X_train, y_train)

# Predict on test set
# Uses the trained patterns to make predictions on new data.
# Without .fit(), .predict() will either crash or return meaningless default values. The code above calls .fit() exactly once
y_pred = model.predict(X_test_pca)


Training XGBoost...


In [10]:
# -----------------------------
# Save Submission
# -----------------------------
submission = pd.DataFrame({'ID': test_df.index, 'y': y_pred})
submission.to_csv('submission.csv', index=False)
print("\n✅ Prediction saved to 'submission.csv'")


✅ Prediction saved to 'submission.csv'
